# Day 6: Audio Features and What They Measure
**Date:** Sunday 28 June 2026

**Conceptual frame:** Spotify's audio features were librosa at scale — the same
signal processing operations, run on 100 million tracks. Understanding what librosa
computes means understanding what 'danceability' and 'valence' actually were.


```{admonition} Conceptual check — before you code
:class: tip

Answer the self-assessment questions for Day 6 before running the cells below.
Questions open in a new tab — come back here when you're done.

**[→ Open Day 6 Quiz](../quizpages/day6_quiz.md)**
```


---
## Part 0: The Score-to-Audio Gap

| | Symbolic (kern) | Audio (recording) |
|--|---|---|
| **Shows** | Pitch, rhythm, intervals | Timbre, dynamics, ornament |
| **Misses** | Performance, feeling | Exact pitch notation, structure |
| **Assumes** | Score is the work | Recording is the work |

```{note}
The Vernadsky Library in Kyiv holds ~466 ethnographic recordings Beregovski made.
They have been digitized but are not freely downloadable. This inaccessibility is
itself data — about whose music gets to be computable.
We use commercial klezmer revival recordings instead.
```


In [ ]:
import requests,zipfile
from pathlib import Path
from collections import Counter
from itertools import islice
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from music21 import converter,note,interval
sns.set_theme(style='whitegrid',font_scale=1.1)
plt.rcParams['figure.figsize']=(10,4)
print('OK')

In [ ]:
CORPUS_DIR=Path('beregovski_corpus');KERN_DIR=CORPUS_DIR/'kern'
if not(KERN_DIR.exists() and list(KERN_DIR.glob('*.krn'))):
    CORPUS_DIR.mkdir(exist_ok=True)
    r=requests.get('https://github.com/shanahdt/mode_in_klezmer/archive/refs/heads/main.zip')
    zp=CORPUS_DIR/'repo.zip';zp.write_bytes(r.content)
    import shutil
    with zipfile.ZipFile(zp) as z:z.extractall(CORPUS_DIR)
    src=list(CORPUS_DIR.glob('mode_in_klezmer-*/kern'))
    if src:
        if KERN_DIR.exists():shutil.rmtree(KERN_DIR)
        shutil.copytree(src[0],KERN_DIR);zp.unlink()
print(f'{len(list(KERN_DIR.glob("*.krn")))} files')

In [ ]:
def load_corpus(kern_dir=KERN_DIR,verbose=True):
    pc2d={7:1,9:2,11:3,0:4,2:5,4:6,6:7,8:2,10:3,1:4,3:5,5:6}
    records,sdict={},{}
    for i,f in enumerate(sorted(Path(kern_dir).glob('*.krn'))):
        if verbose and i%50==0: print(f'  {i+1}...')
        try:
            s=converter.parse(str(f));ns=[n for n in s.flat.notes if isinstance(n,note.Note)]
            pcs=[n.pitch.pitchClass for n in ns]
            records[f.stem]={'tune_id':f.stem,'n_notes':len(ns),
                'pitches':[n.nameWithOctave for n in ns],'pitch_classes':pcs,
                'scale_degrees':[pc2d.get(p,0) for p in pcs],
                'intervals':[interval.Interval(ns[j],ns[j+1]).semitones for j in range(len(ns)-1)]}
            sdict[f.stem]=s
        except: pass
    if verbose: print(f'Loaded {len(records)}')
    return pd.DataFrame(records.values()),sdict

def get_ngrams(seq,n): return list(zip(*[islice(seq,i,None) for i in range(n)]))
print('ready')

In [ ]:
df,streams=load_corpus()
try:
    meta=pd.read_csv('https://raw.githubusercontent.com/shanahdt/mode_in_klezmer/main/metadata.csv')
    df=df.merge(meta,on='tune_id',how='left');print(f'{len(df)} tunes')
except Exception as e: print(e)

In [ ]:
try:
    import librosa,librosa.display
    print(f'librosa {librosa.__version__}')
except:
    import subprocess;subprocess.run([sys.executable,'-m','pip','install','librosa','--quiet'])
    import librosa,librosa.display

In [ ]:
AUDIO_FILE='klezmer_sample.mp3'  # <-- replace with your file
if Path(AUDIO_FILE).exists():
    y,sr=librosa.load(AUDIO_FILE,duration=60)
    print(f'Loaded {AUDIO_FILE}: {len(y)/sr:.1f}s at {sr}Hz')
else:
    print(f'File not found: {AUDIO_FILE}. Using G-major scale demo.')
    sr=22050
    y=np.concatenate([librosa.tone(f,sr=sr,duration=0.3)
                      for f in [392,440,494,523,587,659,740,784]])

In [ ]:
chroma=librosa.feature.chroma_cqt(y=y,sr=sr)
fig,ax=plt.subplots(figsize=(12,4))
librosa.display.specshow(chroma,y_axis='chroma',x_axis='time',sr=sr,ax=ax)
ax.set_title('Chromagram');plt.colorbar(ax.collections[0],ax=ax)
plt.tight_layout();plt.show()

audio_profile=chroma.mean(axis=1);audio_profile/=audio_profile.sum()
pc_names=['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']
print('Audio pitch class profile:')
for n,v in zip(pc_names,audio_profile): print(f'  {n:4s}: {v:.3f}')

In [ ]:
from scipy.stats import pearsonr

all_pcs=[pc for pcs in df['pitch_classes'] for pc in pcs]
pc_counts=Counter(all_pcs);total=sum(pc_counts.values())
score_profile=np.array([pc_counts.get(i,0)/total for i in range(12)])

r,pval=pearsonr(score_profile,audio_profile)
print(f'Pearson r (score vs audio): {r:.3f} (p={pval:.4f})')

x=np.arange(12)
fig,ax=plt.subplots()
ax.bar(x-0.2,score_profile,0.35,label='Score (kern)',color='steelblue',alpha=0.8)
ax.bar(x+0.2,audio_profile,0.35,label='Audio (chroma)',color='coral',alpha=0.8)
ax.set_xticks(x);ax.set_xticklabels(pc_names)
ax.legend();ax.set_title('Score pitch profile vs audio chroma')
plt.tight_layout();plt.show()

In [ ]:
mfccs=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=13)
fig,ax=plt.subplots(figsize=(12,4))
librosa.display.specshow(mfccs,x_axis='time',sr=sr,ax=ax)
ax.set_title('MFCCs — timbre features');plt.colorbar(ax.collections[0],ax=ax)
plt.tight_layout();plt.show()
print('MFCCs describe spectral envelope (timbre), not pitch.')

---
## Day 6 Exercise: Score-to-Audio Gap

```{admonition} Exercise
Write 200–250 words: what does the Pearson correlation between score and audio profiles mean?
Does the gap (or agreement) reflect a limitation of the audio method, the score,
the specific recording, or something else? What would you need to know to decide?
```


### Your answer

*(200–250 words)*


---
## Project Log — Entry 6

> *Score-to-audio correlation: [r].*  
> *The gap tells me [Y] about [encoding/recording/method].*  
> *Audio reveals: [X]. Score shows what audio obscures: [Y].*

*(100–150 words)*
